In [5]:
%pip install pandas numpy matplotlib openpyxl scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 1.7 MB/s eta 0:00:05
   --- ------------------------------------ 0.8/8.2 MB 1.7 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.2 MB 1.6 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.8 MB/s eta 0:00:04
   ---------- ----------------------------- 2.1/8.2 MB 1.9 MB/s eta 0:00:04
   ----------- ---------------------------- 2.4/8.2 MB 1.7 MB/s eta 0:00:04
   -------------- ------------------------- 2.9/8.2 MB 1.7 MB/s eta 0:00:04
   ---------------- ----------------------- 3.4/8.2 MB 1.8 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.2 MB 1.8 MB/s eta 0:00:03
   ------------------- -------------------- 3.9/8.2 MB 1.7 MB/s eta 0:00:03
   --------------------- --------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [10]:
desi = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Desi_talep.xlsx")

In [12]:
print(desi.columns.tolist())
print("\nİlk 3 satır:")
print(desi.head(3))

['Çıkış Transfer Merkezi', 'Varış Transfer Merkezi', 'Tarih', 'Toplam Desi']

İlk 3 satır:
  Çıkış Transfer Merkezi Varış Transfer Merkezi      Tarih  Toplam Desi
0                 Mersin              Eskişehir 2026-01-01       379.50
1                Kocaeli              Şanlıurfa 2026-01-01      1239.48
2                Kocaeli               Tekirdağ 2026-01-01      2852.70


In [13]:
# Sütunları rename 
desi.columns = ['cikis', 'varis', 'tarih', 'desi']

# tarih'i datetime'a çevir
desi['tarih'] = pd.to_datetime(desi['tarih'])

# Kontrol
print("Tipler:")
print(desi.dtypes)
print("\nİlk 5 satır:")
print(desi.head())

Tipler:
cikis            object
varis            object
tarih    datetime64[ns]
desi            float64
dtype: object

İlk 5 satır:
     cikis      varis      tarih     desi
0   Mersin  Eskişehir 2026-01-01   379.50
1  Kocaeli  Şanlıurfa 2026-01-01  1239.48
2  Kocaeli   Tekirdağ 2026-01-01  2852.70
3    Sivas  Eskişehir 2026-01-01   638.60
4  Kocaeli      Sivas 2026-01-01  1001.60


In [14]:
# ================================
# FEATURE ENGINEERING
# ================================

# 1. Tarihten feature'lar
desi['day_of_week'] = desi['tarih'].dt.dayofweek      # 0=Pazartesi, 6=Pazar
desi['is_monday']   = (desi['day_of_week'] == 0).astype(int)  # Pazartesi mi?
desi['is_weekend']  = (desi['day_of_week'] >= 5).astype(int)  # Hafta sonu mu?
desi['gun_sirasi']  = (desi['tarih'] - desi['tarih'].min()).dt.days  # 0,1,2,3...

# 2. Her rota için lag ve rolling feature
desi = desi.sort_values(['cikis','varis','tarih']).reset_index(drop=True)

desi['lag_7']         = desi.groupby(['cikis','varis'])['desi'].shift(7)
desi['lag_14']        = desi.groupby(['cikis','varis'])['desi'].shift(14)
desi['rolling_mean_7']= desi.groupby(['cikis','varis'])['desi'].transform(
                            lambda x: x.shift(1).rolling(7).mean())

print("Feature'lar eklendi:")
print(desi.columns.tolist())
print("\nİlk 10 satır:")
print(desi.head(10))
print("\nEksik değer (lag yüzünden normal):")
print(desi.isnull().sum())

Feature'lar eklendi:
['cikis', 'varis', 'tarih', 'desi', 'day_of_week', 'is_monday', 'is_weekend', 'gun_sirasi', 'lag_7', 'lag_14', 'rolling_mean_7']

İlk 10 satır:
       cikis    varis      tarih      desi  day_of_week  is_monday  \
0  Balıkesir  Bilecik 2026-01-01    327.36            3          0   
1  Balıkesir  Bilecik 2026-01-02   8399.76            4          0   
2  Balıkesir  Bilecik 2026-01-03   4464.00            5          0   
3  Balıkesir  Bilecik 2026-01-04    379.44            6          0   
4  Balıkesir  Bilecik 2026-01-05  10535.04            0          1   
5  Balıkesir  Bilecik 2026-01-06   8511.36            1          0   
6  Balıkesir  Bilecik 2026-01-07   6219.84            2          0   
7  Balıkesir  Bilecik 2026-01-08   7640.88            3          0   
8  Balıkesir  Bilecik 2026-01-09   6554.64            4          0   
9  Balıkesir  Bilecik 2026-01-10   3251.28            5          0   

   is_weekend  gun_sirasi    lag_7  lag_14  rolling_mean_7  
0  

BASELINE MODEL

In [29]:
# ================================
# BLOK 1: BASELINE MODEL
# Son 4 haftanın aynı gününün ortalaması
# ================================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Veri yükle
desi = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Desi_talep.xlsx")
desi.columns = ['cikis', 'varis', 'tarih', 'desi']
desi['tarih'] = pd.to_datetime(desi['tarih'])
desi['day_of_week'] = desi['tarih'].dt.dayofweek

rotalar = desi[['cikis','varis']].drop_duplicates().reset_index(drop=True)

# --- Validation (4-10 Mayıs) için hata ölçümü ---
val_sonuclar = []
for _, rota in rotalar.iterrows():
    cikis, varis = rota['cikis'], rota['varis']
    rota_df = desi[(desi['cikis']==cikis) & (desi['varis']==varis)]
    
    for tarih in pd.date_range('2026-05-04', '2026-05-10'):
        gun = tarih.dayofweek
        gecmis = rota_df[
            (rota_df['day_of_week'] == gun) &
            (rota_df['tarih'] < tarih)
        ].tail(4)
        
        tahmin = gecmis['desi'].mean() if len(gecmis) > 0 else rota_df['desi'].mean()
        val_sonuclar.append({
            'tarih': tarih, 'cikis': cikis,
            'varis': varis, 'tahmin': tahmin
        })

val_df_gercek = desi[(desi['tarih'] >= '2026-05-04') & 
                     (desi['tarih'] <= '2026-05-10')]
val_baseline_df = pd.DataFrame(val_sonuclar)
merged = val_df_gercek.merge(val_baseline_df, on=['tarih','cikis','varis'])
mae = mean_absolute_error(merged['desi'], merged['tahmin'])
print(f"Baseline Validation MAE: {mae:.1f} desi")

# --- Asıl Tahmin (11-17 Mayıs) ---
tahmin_sonuclar = []
for _, rota in rotalar.iterrows():
    cikis, varis = rota['cikis'], rota['varis']
    rota_df = desi[(desi['cikis']==cikis) & (desi['varis']==varis)]
    
    for tarih in pd.date_range('2026-05-11', '2026-05-17'):
        gun = tarih.dayofweek
        gecmis = rota_df[rota_df['day_of_week'] == gun].tail(4)
        
        tahmin = gecmis['desi'].mean() if len(gecmis) > 0 else rota_df['desi'].mean()
        tahmin_sonuclar.append({
            'tarih': tarih, 'cikis': cikis,
            'varis': varis, 'tahmin': round(tahmin, 2)
        })

baseline_df = pd.DataFrame(tahmin_sonuclar)
print(f"\nTahmin shape: {baseline_df.shape}")
print(baseline_df.head(7).to_string())

Baseline Validation MAE: 3891.1 desi

Tahmin shape: (623, 4)
       tarih   cikis      varis   tahmin
0 2026-05-11  Mersin  Eskişehir  6782.93
1 2026-05-12  Mersin  Eskişehir  4374.37
2 2026-05-13  Mersin  Eskişehir  3896.20
3 2026-05-14  Mersin  Eskişehir  3649.52
4 2026-05-15  Mersin  Eskişehir  2932.27
5 2026-05-16  Mersin  Eskişehir  3344.66
6 2026-05-17  Mersin  Eskişehir   333.96


In [30]:
# Tahmin dosyasını kaydet
tahmin_df = baseline_df.rename(columns={
    'cikis': 'Çıkış Transfer Merkezi',
    'varis': 'Varış Transfer Merkezi',
    'tarih': 'Tarih',
    'tahmin': 'Tahmin Edilen Desi'
})

tahmin_df['Tahmin Edilen Desi'] = tahmin_df['Tahmin Edilen Desi'].clip(lower=0).round(2)

tahmin_df.to_excel(
    r"C:\Users\semanur\Desktop\HB_Yarisma\data2\tahminlenen_talep.xlsx", 
    index=False
)
print("✅ Tahmin dosyası kaydedildi!")
print(f"Toplam satır: {len(tahmin_df)}")
print(tahmin_df.head(7).to_string())

✅ Tahmin dosyası kaydedildi!
Toplam satır: 623
       Tarih Çıkış Transfer Merkezi Varış Transfer Merkezi  Tahmin Edilen Desi
0 2026-05-11                 Mersin              Eskişehir             6782.93
1 2026-05-12                 Mersin              Eskişehir             4374.37
2 2026-05-13                 Mersin              Eskişehir             3896.20
3 2026-05-14                 Mersin              Eskişehir             3649.52
4 2026-05-15                 Mersin              Eskişehir             2932.27
5 2026-05-16                 Mersin              Eskişehir             3344.66
6 2026-05-17                 Mersin              Eskişehir              333.96


In [32]:
# ================================
# BASELINE METRİK KARŞILAŞTIRMASI
# MAE, MAPE, WMAPE
# ================================

# Validation seti ve baseline tahminleri hazır olmalı
# merged değişkeni: 'desi' (gerçek) ve 'tahmin' sütunları

import numpy as np

gercek = merged['desi'].values
tahmin = merged['tahmin'].values

# MAE
mae = np.mean(np.abs(gercek - tahmin))

# MAPE - sıfıra çok yakın gerçek değerler hatayı patlatır
# o yüzden gerçek > 100 olanları alalım
maske = gercek > 100
mape = np.mean(np.abs((gercek[maske] - tahmin[maske]) / gercek[maske])) * 100

# WMAPE - toplam hatayı toplam gerçeğe böl
wmape = np.sum(np.abs(gercek - tahmin)) / np.sum(gercek) * 100

print("=" * 40)
print("BASELINE MODEL METRİKLERİ")
print("=" * 40)
print(f"MAE   : {mae:.1f} desi")
print(f"MAPE  : %{mape:.1f}  (100 desi üzeri rotalar)")
print(f"WMAPE : %{wmape:.1f}  (ağırlıklı, en güvenilir)")
print("=" * 40)
print("\nYorum:")
print(f"WMAPE %20 altında → {'İyi ✅' if wmape < 20 else 'Geliştirilebilir ⚠️'}")
print(f"WMAPE %10 altında → {'Çok iyi ✅' if wmape < 10 else 'Henüz değil'}")

BASELINE MODEL METRİKLERİ
MAE   : 3891.1 desi
MAPE  : %62.8  (100 desi üzeri rotalar)
WMAPE : %25.6  (ağırlıklı, en güvenilir)

Yorum:
WMAPE %20 altında → Geliştirilebilir ⚠️
WMAPE %10 altında → Henüz değil


In [31]:
# Sağlama - Mersin→Eskişehir Pazartesi tahmini doğru mu?
kontrol = desi[(desi['cikis']=='Mersin') & (desi['varis']=='Eskişehir')]

# Geçmişteki tüm Pazartesiler
pazartesiler = kontrol[kontrol['day_of_week'] == 0].tail(4)
print("Son 4 Pazartesi (Mersin→Eskişehir):")
print(pazartesiler[['tarih','desi']])
print(f"\nOrtalamaları: {pazartesiler['desi'].mean():.2f}")
print(f"Tahmin dosyasındaki 11 Mayıs değeri: 6782.93")

Son 4 Pazartesi (Mersin→Eskişehir):
           tarih     desi
8474  2026-04-13  7397.72
9068  2026-04-20  7306.64
9756  2026-04-27  6127.66
10313 2026-05-04  6299.70

Ortalamaları: 6782.93
Tahmin dosyasındaki 11 Mayıs değeri: 6782.93


In [16]:
%pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 2.6 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ŞİMDİ LIGHTGBM İLE TAHMİNLEME

In [25]:
# ================================
# BLOK 2: LightGBM MODEL
# (Bu veri için baseline'dan kötü çıktı - referans için saklıyoruz)
# ================================

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# Feature engineering
desi['is_monday']  = (desi['tarih'].dt.dayofweek == 0).astype(int)
desi['is_weekend'] = (desi['tarih'].dt.dayofweek >= 5).astype(int)
desi['gun_sirasi'] = (desi['tarih'] - desi['tarih'].min()).dt.days

desi = desi.sort_values(['cikis','varis','tarih']).reset_index(drop=True)
desi['lag_7']          = desi.groupby(['cikis','varis'])['desi'].shift(7)
desi['lag_14']         = desi.groupby(['cikis','varis'])['desi'].shift(14)
desi['rolling_mean_7'] = desi.groupby(['cikis','varis'])['desi'].transform(
                             lambda x: x.shift(1).rolling(7).mean())

le_cikis = LabelEncoder()
le_varis  = LabelEncoder()
desi['cikis_enc'] = le_cikis.fit_transform(desi['cikis'])
desi['varis_enc']  = le_varis.fit_transform(desi['varis'])

# Train / Validation
train_df = desi[desi['tarih'] < '2026-05-04'].dropna()
val_df   = desi[(desi['tarih'] >= '2026-05-04') &
                (desi['tarih'] <= '2026-05-10')].dropna()

features = ['day_of_week','is_monday','is_weekend','gun_sirasi',
            'lag_7','lag_14','rolling_mean_7','cikis_enc','varis_enc']

model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                           num_leaves=31, random_state=42, verbose=-1)
model.fit(train_df[features], train_df['desi'])

val_pred = model.predict(val_df[features])
mae_lgbm = mean_absolute_error(val_df['desi'], val_pred)

print(f"LightGBM Validation MAE : {mae_lgbm:.1f} desi")
print(f"Baseline Validation MAE : {mae:.1f} desi")
print(f"\nKullanılacak model: {'Baseline ✅' if mae < mae_lgbm else 'LightGBM ✅'}")

LightGBM Validation MAE : 5828.8 desi
Baseline Validation MAE : 3891.1 desi

Kullanılacak model: Baseline ✅


In [26]:
# ================================
# BLOK 2 GELİŞTİRİLMİŞ: LightGBM
# ================================

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# Temiz başla
desi2 = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Desi_talep.xlsx")
desi2.columns = ['cikis', 'varis', 'tarih', 'desi']
desi2['tarih'] = pd.to_datetime(desi2['tarih'])
desi2 = desi2.sort_values(['cikis','varis','tarih']).reset_index(drop=True)

# Tarih feature'ları
desi2['day_of_week'] = desi2['tarih'].dt.dayofweek
desi2['is_monday']   = (desi2['day_of_week'] == 0).astype(int)
desi2['is_weekend']  = (desi2['day_of_week'] >= 5).astype(int)
desi2['gun_sirasi']  = (desi2['tarih'] - desi2['tarih'].min()).dt.days
desi2['month']       = desi2['tarih'].dt.month
desi2['week_of_year']= desi2['tarih'].dt.isocalendar().week.astype(int)

# Lag feature'ları
grp = desi2.groupby(['cikis','varis'])['desi']
desi2['lag_1']          = grp.shift(1)
desi2['lag_7']          = grp.shift(7)
desi2['lag_14']         = grp.shift(14)
desi2['lag_28']         = grp.shift(28)
desi2['rolling_mean_7'] = grp.transform(lambda x: x.shift(1).rolling(7).mean())
desi2['rolling_mean_28']= grp.transform(lambda x: x.shift(1).rolling(28).mean())
desi2['rolling_std_7']  = grp.transform(lambda x: x.shift(1).rolling(7).std())

# Encoding
le_c = LabelEncoder()
le_v = LabelEncoder()
desi2['cikis_enc'] = le_c.fit_transform(desi2['cikis'])
desi2['varis_enc'] = le_v.fit_transform(desi2['varis'])

# Train / Validation
train2 = desi2[desi2['tarih'] < '2026-05-04'].dropna()
val2   = desi2[(desi2['tarih'] >= '2026-05-04') &
               (desi2['tarih'] <= '2026-05-10')].dropna()

features2 = ['day_of_week','is_monday','is_weekend','gun_sirasi',
             'month','week_of_year',
             'lag_1','lag_7','lag_14','lag_28',
             'rolling_mean_7','rolling_mean_28','rolling_std_7',
             'cikis_enc','varis_enc']

model2 = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03,
                            num_leaves=31, random_state=42, verbose=-1)
model2.fit(train2[features2], train2['desi'])

val_pred2 = model2.predict(val2[features2])
mae_lgbm2 = mean_absolute_error(val2['desi'], val_pred2)

print(f"Geliştirilmiş LightGBM MAE : {mae_lgbm2:.1f} desi")
print(f"Baseline MAE               : 3891.1 desi")
print(f"\nSonuç: {'LightGBM kazandı ✅' if mae_lgbm2 < 3891.1 else 'Baseline hala iyi'}")

# Feature önem sırası
feat_imp = pd.Series(model2.feature_importances_, index=features2).sort_values(ascending=False)
print("\nFeature önem sırası:")
print(feat_imp)

Geliştirilmiş LightGBM MAE : 5859.5 desi
Baseline MAE               : 3891.1 desi

Sonuç: Baseline hala iyi

Feature önem sırası:
gun_sirasi         4488
lag_1              3950
lag_7              3298
lag_14             2867
rolling_std_7      2708
lag_28             2701
rolling_mean_28    2591
rolling_mean_7     2389
day_of_week        2080
cikis_enc          1717
varis_enc          1211
week_of_year          0
month                 0
is_monday             0
is_weekend            0
dtype: int32


Tamin modelini geliştirmek için daha fazla feature eklenebilir, hiperparametre optimizasyonu yapılabilir veya farklı modeller denenebilir. Ancak, mevcut veri seti ve özellikler göz önüne alındığında, baseline modelin performansı oldukça iyi görünüyor.